# Speech (§4.1 ASR + §4.2 TTS): Real Smoke Test

Covers task backlog §4.1-4.2. Real calls throughout, not mocked: a real OpenAI TTS synthesis call feeds real audio bytes into a real local `faster-whisper` transcription call, so this is a genuine round trip through both modules' actual I/O boundary, not two isolated unit tests that happen to pass independently.

**Provider choices (confirmed with the user before adding as new dependencies):**
- **ASR:** `faster-whisper`, local/free, no API key or billing -- same cost-conscious reasoning as §1.3's local sentence-transformers embeddings.
- **TTS:** OpenAI TTS API, paid but cheap (~$15/1M chars) -- real spend, kept small and disclosed below (see the cost summary at the end).

**Real bug found and fixed while writing this, not assumed working:** `tts.synthesize()`'s length-budget correction originally used `wave.getnframes()` to measure clip duration, which trusts a header field. OpenAI's streamed `"wav"` response sets that field to the streaming placeholder `0xFFFFFFFF` (length unknown at encode time) -- turning a real ~2s clip into a reported ~24.8 *day* duration and wrongly forcing every synthesis to the 4x speed cap. Caught by actually running this round trip and printing the real numbers (a duration of `89478.49s` on a one-sentence clip is obviously wrong on inspection) -- a test that only asserted `duration_seconds > 0` would have passed anyway and missed it. Fixed by computing duration from the actual received byte count instead of the declared header field; see `_wav_duration_seconds()` in `tts.py` for the full explanation.

**Known limitations, stated not hidden:**
- The multilingual smoke test below uses **synthesized** (TTS-generated) non-English audio, not real accented human speech -- it verifies the ASR pipeline handles non-English input and language detection correctly, not real-world accent robustness.
- The length-budget correction is a single regeneration pass, not an iterative loop: OpenAI's `speed` parameter doesn't scale duration perfectly linearly (there's fixed leading/trailing padding that doesn't compress), so a corrected clip can land a few hundred ms to ~1-2s over the nominal 15s budget rather than landing exactly on it. A second corrective pass would tighten this further at the cost of an extra call for the rare over-budget case; not implemented since one pass gets close enough for a demo, same "stated rough edge, not silently perfected" posture as §2.5's `enforce_length()`.

In [1]:
# !pip install faster-whisper openai python-dotenv

## Setup

In [2]:
import os
import time

import asr
import speech_config as config
import tts

TMP_DIR = os.environ.get("TMPDIR", "/tmp")
print("asr.py / tts.py imported OK")
print("ASR provider:", config.ASR_PROVIDER, "model:", config.ASR_MODEL_SIZE)
print("TTS provider:", config.TTS_PROVIDER, "model:", config.TTS_MODEL)

asr.py / tts.py imported OK
ASR provider: faster-whisper model: base
TTS provider: openai model: tts-1


## §4.1 + §4.2 round trip: real TTS synthesis -> real ASR transcription

Text modeled on a real Answerer (§2.5) output shape (price + product name), synthesized
for real via OpenAI TTS, written to a real WAV file, then transcribed back for real via
local `faster-whisper`. Checks: synthesis succeeds within the 15s budget at normal speed,
the WAV duration measurement is sane (not the `wave.getnframes()` bug above), and the
round-tripped transcript is a close match to the original text.

In [3]:
ANSWER_TEXT = (
    "The cheapest insulated lunch box under twenty dollars is the Wildkin kids "
    "lunch bag, priced at twelve ninety nine."
)
print("word count:", len(ANSWER_TEXT.split()))

t0 = time.time()
tts_result = tts.synthesize(ANSWER_TEXT)
tts_elapsed = time.time() - t0
print(f"TTS: {tts_elapsed:.2f}s wall, duration={tts_result.duration_seconds:.2f}s, "
      f"speed_used={tts_result.speed_used}, bytes={len(tts_result.audio_bytes)}")
assert tts_result.duration_seconds <= config.TTS_MAX_SECONDS, "over the 15s budget at normal speed"
assert tts_result.speed_used == 1.0, "unexpected speed correction on a short answer"

wav_path = os.path.join(TMP_DIR, "speech_smoke_roundtrip.wav")
with open(wav_path, "wb") as f:
    f.write(tts_result.audio_bytes)

t0 = time.time()
asr_result = asr.transcribe(wav_path)
asr_elapsed = time.time() - t0
print(f"ASR: {asr_elapsed:.2f}s wall, language={asr_result.language} (p={asr_result.language_probability:.2f})")
print("original: ", ANSWER_TEXT)
print("roundtrip:", asr_result.text)
print(f"{len(asr_result.segments)} segment(s):")
for s in asr_result.segments:
    print(f"  [{s.start:.2f}-{s.end:.2f}] {s.text}")

assert asr_result.language == "en"
# Loose containment check, not exact match -- Whisper normalizes spoken numbers to
# digits ("twenty dollars" -> "$20"), so exact string equality isn't the right bar.
assert "wildkin" in asr_result.text.lower()
assert "12.99" in asr_result.text or "twelve" in asr_result.text.lower()
print("\nround trip: OK")

word count: 19


TTS: 3.06s wall, duration=7.19s, speed_used=1.0, bytes=345044


ASR: 3.70s wall, language=en (p=1.00)
original:  The cheapest insulated lunch box under twenty dollars is the Wildkin kids lunch bag, priced at twelve ninety nine.
roundtrip: The cheapest insulated lunchbox under $20 is the Wildkin Kids lunch bag, priced at $12.99.
1 segment(s):
  [0.00-7.00] The cheapest insulated lunchbox under $20 is the Wildkin Kids lunch bag, priced at $12.99.

round trip: OK


## §4.2 length-budget correction, exercised for real

A ~54-word answer (well over the Answerer's own ~40-word budget, deliberately, to force
`synthesize()`'s duration>budget correction path) should regenerate once at a faster
`speed` and land close to the 15s budget -- not just have that logic present and untested.

In [4]:
LONG_TEXT = (
    "Based on your budget and preferences, I would recommend considering the Wildkin "
    "kids insulated lunch box at twelve ninety nine, or alternatively the Bentgo "
    "stainless steel bento box at eighteen fifty if you would prefer something more "
    "durable, leak proof, and better suited for daily school use throughout the entire "
    "year, including field trips."
)
print("word count:", len(LONG_TEXT.split()))

long_result = tts.synthesize(LONG_TEXT)
print(f"duration={long_result.duration_seconds:.2f}s, speed_used={long_result.speed_used}")

assert long_result.speed_used > 1.0, "expected the correction path to trigger on this long text"
# One correction pass, not exact -- see the intro cell's note on OpenAI's `speed`
# parameter not scaling duration perfectly linearly. Loose tolerance, stated as such.
assert long_result.duration_seconds <= config.TTS_MAX_SECONDS + 2.0, "still well over budget after correction"
print("length-budget correction: OK (triggered for real, landed close to budget)")

word count: 54


duration=15.03s, speed_used=1.4349999999999998
length-budget correction: OK (triggered for real, landed close to budget)


## §4.1 multilingual smoke test

Per the intro cell's stated limitation: this is synthesized (TTS-generated) non-English
audio, not real accented human speech -- no real multilingual/accented recordings were
on hand, so this checks that the ASR pipeline correctly handles non-English input and
detects language, not real-world accent robustness.

In [5]:
MULTILINGUAL_CASES = [
    ("es", "¿Cuál es la lonchera aislada más barata que tienes disponible por menos de veinte dólares?"),
    ("fr", "Quel est le prix de la boîte à lunch isotherme la moins chère disponible en ce moment?"),
]

for expected_lang, text in MULTILINGUAL_CASES:
    result = tts.synthesize(text, max_seconds=30.0)  # relaxed budget -- not testing length here
    path = os.path.join(TMP_DIR, f"speech_smoke_{expected_lang}.wav")
    with open(path, "wb") as f:
        f.write(result.audio_bytes)
    tr = asr.transcribe(path)
    print(f"expected={expected_lang} detected={tr.language} (p={tr.language_probability:.2f})")
    print("  text:", tr.text)
    assert tr.language == expected_lang, f"language mismatch: expected {expected_lang}, got {tr.language}"

print("\nmultilingual smoke test: OK")

expected=es detected=es (p=1.00)
  text: ¿Cuál es la lonchera islada más barata que tienes disponible por menos de 20 dólares?


expected=fr detected=fr (p=0.99)
  text: Quel est le prix de la boîte à l'unche isoterme la moins chère disponible en ce moment ?

multilingual smoke test: OK


## Error handling (0 additional cost)

`ASRError`/`TTSError` are meant to be typed exceptions the UI (§5, not yet built) can
catch for its own "TTS failure -> text-only fallback" requirement (§4.2/§5.3) -- checked
directly here rather than assumed to work because the `try`/`except` blocks look right.
None of these make a real API call or load the Whisper model.

In [6]:
# ASRError on a missing file
try:
    asr.transcribe(os.path.join(TMP_DIR, "does_not_exist_12345.wav"))
    print("FAIL: expected ASRError")
except asr.ASRError as e:
    print("ASRError raised correctly:", e)

# TTSError on empty/whitespace-only text
try:
    tts.synthesize("   ")
    print("FAIL: expected TTSError")
except tts.TTSError as e:
    print("TTSError raised correctly:", e)

# TTSError on an unimplemented provider -- provider-swappable by construction (§2.6's
# get_chat_model() pattern), but only the openai path is actually built; this checks the
# "not implemented" branch fails loud and typed rather than silently no-op-ing
_orig_provider = tts.TTS_PROVIDER
tts.TTS_PROVIDER = "elevenlabs"
try:
    tts.synthesize("hello")
    print("FAIL: expected TTSError")
except tts.TTSError as e:
    print("TTSError raised correctly:", e)
finally:
    tts.TTS_PROVIDER = _orig_provider

print("\nerror handling: OK")

ASRError raised correctly: Audio file not found: /var/folders/dt/hz0jqyys0fz2ksjdx5kkxm800000gn/T/does_not_exist_12345.wav
TTSError raised correctly: Cannot synthesize empty text
TTSError raised correctly: TTS_PROVIDER='elevenlabs' has no implemented synthesis path (only 'openai' is built)

error handling: OK


## Cost/timing summary

Every real OpenAI TTS call this notebook made, disclosed exactly (not estimated from a
call count) -- ASR is $0 throughout (local, free). The long-text correction case makes
two real TTS calls (original attempt + one corrected regeneration), which is why it's
counted twice below.

In [7]:
# Real OpenAI TTS calls made in this notebook, one row per call (long text is 2 calls:
# the original over-budget attempt + the corrected regeneration).
tts_calls = [
    ("round trip", ANSWER_TEXT),
    ("long text (original)", LONG_TEXT),
    ("long text (corrected regen)", LONG_TEXT),
] + [(f"multilingual ({lang})", text) for lang, text in MULTILINGUAL_CASES]

total_chars = sum(len(text) for _, text in tts_calls)
# $15 / 1,000,000 chars, per config.py's TTS_MODEL=tts-1 pricing
est_cost_usd = total_chars / 1_000_000 * 15.0

print(f"Real OpenAI TTS calls: {len(tts_calls)}")
for label, text in tts_calls:
    print(f"  - {label}: {len(text)} chars")
print(f"Total characters sent to TTS: {total_chars}")
print(f"Estimated cost at tts-1 pricing (~$15/1M chars): ${est_cost_usd:.5f}")
print("Real local faster-whisper transcriptions (ASR, $0 -- local compute only): 3 "
      "(round trip + 2 multilingual). The error-path 'file not found' case raises before "
      "the model is even loaded, so it costs nothing, not even local compute.")

Real OpenAI TTS calls: 5
  - round trip: 114 chars
  - long text (original): 345 chars
  - long text (corrected regen): 345 chars
  - multilingual (es): 90 chars
  - multilingual (fr): 86 chars
Total characters sent to TTS: 980
Estimated cost at tts-1 pricing (~$15/1M chars): $0.01470
Real local faster-whisper transcriptions (ASR, $0 -- local compute only): 3 (round trip + 2 multilingual). The error-path 'file not found' case raises before the model is even loaded, so it costs nothing, not even local compute.
